# AI-Powered Coding Tutor — Giai đoạn 3: Kaggle GPU Embedding Pilot

Notebook này triển khai thử nghiệm **Pilot (160 chunks)** độc lập hoàn toàn trên Kaggle Cloud GPU:
- **Pilot A:** 120 TEXT chunks chạy mô hình `Qwen/Qwen3-Embedding-0.6B` (Dimension: 1.024, context 32k).
- **Pilot B:** 40 CODE chunks chạy mô hình `jinaai/jina-embeddings-v2-base-code` (Dimension: 768, context 8k).

### Quy tắc an toàn & Kiến trúc:
1. **Tuân thủ quy tắc Modality:** Ngưỡng `threshold = 0.50` (Toàn bộ dataset: 4.293 TEXT và 937 CODE).
2. **Thực thi tuần tự (Sequential Loading):** Nạp Model 1 -> Chạy -> Giải phóng hoàn toàn VRAM -> Nạp Model 2.
3. **Độ bền Checkpoint (Cross-session & Intra-session):** Tự động khôi phục và đối chiếu fingerprint, model_id, dimension chống ghi đè.
4. **Kiểm tra chất lượng nghiêm ngặt (Strict QA):** Kiểm tra Dimension, L2 Norm (~1.0), không NaN/Inf, bảo toàn nguyên vẹn 100% metadata.

In [ ]:
# =============================================================================
# 1. KIỂM TRA PHẦN CỨNG GPU & MÔI TRƯỜNG KAGGLE
# =============================================================================
import os
import sys
import torch
import gc

print("=== BÁO CÁO MÔI TRƯỜNG KAGGLE ===")
print(f"Python Version: {sys.version.split()[0]}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    raise SystemError("❌ LỖI: GPU chưa được bật! Vui lòng vào Notebook options (bên phải) -> Accelerator -> Chọn GPU T4 x2 hoặc GPU T4.")

gpu_count = torch.cuda.device_count()
gpu_name = torch.cuda.get_device_name(0)
vram_total = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)

print(f"✅ ĐÃ KẾT NỐI GPU THÀNH CÔNG:")
print(f"  • GPU Model   : {gpu_name} (Số lượng: {gpu_count})")
print(f"  • Tổng VRAM   : {vram_total:.2f} GB")

# Reset bộ nhớ GPU
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
print("🧹 Bộ nhớ GPU đã được dọn sạch và sẵn sàng.")


In [ ]:
# =============================================================================
# 2. CÀI ĐẶT CÁC THƯ VIỆN CẦN THIẾT TRÊN MÔI TRƯỜNG KAGGLE
# =============================================================================
!pip install -q --upgrade transformers sentence-transformers accelerate einops
print("✅ Cài đặt thư viện Deep Learning hoàn tất!")


In [ ]:
# =============================================================================
# 3. THIẾT LẬP ĐƯỜNG DẪN INPUT & OUTPUT
# =============================================================================
from pathlib import Path

# Tự động phát hiện file input từ Kaggle Dataset
candidate_paths = [
    Path("/kaggle/input/ai-coding-tutor-chunks/all_chunks.json"),
    Path("/kaggle/input/ai-coding-tutor-chunks/pilot_chunks_160.json"),
    Path("/kaggle/input/all_chunks.json"),
    Path("data/06_rag_dataset/chunks/all_chunks.json"),
    Path("data/06_rag_dataset/chunks/pilot_chunks_160.json")
]

INPUT_CHUNK_FILE = None
for cp in candidate_paths:
    if cp.exists():
        INPUT_CHUNK_FILE = cp
        break

if INPUT_CHUNK_FILE is None:
    # Liệt kê thư mục /kaggle/input để hỗ trợ debug đường dẫn
    print("Danh sách file trong /kaggle/input:")
    for root, dirs, files in os.walk("/kaggle/input"):
        for f in files:
            print(os.path.join(root, f))
    raise FileNotFoundError("❌ Không tìm thấy file chunk trong /kaggle/input/. Vui lòng kiểm tra đã Add Input Dataset chưa!")

print(f"📂 File input đã tìm thấy: {INPUT_CHUNK_FILE} ({INPUT_CHUNK_FILE.stat().st_size:,} bytes)")

# Thiết lập các thư mục làm việc tại /kaggle/working/
WORKING_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("data/08_rag_dataset")
CHECKPOINT_DIR = WORKING_DIR / "checkpoints"
EMBEDDINGS_DIR = WORKING_DIR / "embeddings"
REPORTS_DIR = WORKING_DIR / "reports"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
EMBEDDINGS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"📁 Thư mục checkpoints: {CHECKPOINT_DIR}")
print(f"📁 Thư mục embeddings : {EMBEDDINGS_DIR}")
print(f"📁 Thư mục reports    : {REPORTS_DIR}")


In [ ]:
# =============================================================================
# 4. NHÚNG MÃ NGUỒN EMBEDDING FRAMEWORK TRONG NOTEBOOK
# =============================================================================
import re
import math
import json
import time
import random
import datetime
from collections import defaultdict
from enum import Enum
from typing import List, Dict, Tuple, Set, Optional

class Modality(str, Enum):
    TEXT = "text"
    CODE = "code"

class ModalityRouter:
    """Định tuyến Modality theo quy tắc 50% Dominance."""
    def __init__(self, code_ratio_threshold: float = 0.50):
        self.code_ratio_threshold = code_ratio_threshold
        self._code_pattern = re.compile(r"```[a-zA-Z0-9_\-\.\+]*\s*([\s\S]*?)```", re.MULTILINE)

    def calculate_code_ratio(self, text: str) -> float:
        if not text:
            return 0.0
        matches = self._code_pattern.findall(text)
        if not matches:
            return 0.0
        return sum(len(m) for m in matches) / len(text)

    def classify_chunk(self, chunk: dict) -> Tuple[Modality, float]:
        metadata = chunk.get("metadata", {})
        content_type = metadata.get("content_type", "")
        has_code = metadata.get("has_code", False)
        text = chunk.get("text", "")

        if content_type == "code_example":
            return Modality.CODE, self.calculate_code_ratio(text)
        if has_code:
            ratio = self.calculate_code_ratio(text)
            if ratio >= self.code_ratio_threshold:
                return Modality.CODE, ratio
            return Modality.TEXT, ratio
        return Modality.TEXT, 0.0

    def route(self, chunks: List[dict]) -> Dict[Modality, List[dict]]:
        res = {Modality.TEXT: [], Modality.CODE: []}
        for c in chunks:
            mod, _ = self.classify_chunk(c)
            res[mod].append(c)
        return res

class PilotSampler:
    """Lấy mẫu Pilot có tính đại diện và tái lập 100% bằng seed cố định."""
    def __init__(self, seed: int = 42, router: Optional[ModalityRouter] = None):
        self.seed = seed
        self.router = router or ModalityRouter(code_ratio_threshold=0.50)

    def sample(self, chunks: List[dict], text_limit: int = 120, code_limit: int = 40) -> Dict[str, List[dict]]:
        rng = random.Random(self.seed)
        routed = self.router.route(chunks)
        text_chunks = routed[Modality.TEXT]
        code_chunks = routed[Modality.CODE]

        # Stratified sampling cho TEXT theo source_file
        text_by_file = defaultdict(list)
        for c in text_chunks:
            text_by_file[c.get("metadata", {}).get("source_file", "unknown")].append(c)

        sampled_text = []
        total_text = len(text_chunks)
        remaining = text_limit
        for sfile, flist in sorted(text_by_file.items()):
            cnt = max(1, round(len(flist) / total_text * text_limit))
            cnt = min(cnt, len(flist), remaining)
            fcopy = list(flist)
            rng.shuffle(fcopy)
            sampled_text.extend(fcopy[:cnt])
            remaining -= cnt

        if len(sampled_text) < text_limit:
            used = {x["id"] for x in sampled_text}
            rem = [c for c in text_chunks if c["id"] not in used]
            rng.shuffle(rem)
            sampled_text.extend(rem[:text_limit - len(sampled_text)])
        elif len(sampled_text) > text_limit:
            sampled_text = sampled_text[:text_limit]

        # Cân bằng CODE: 30 code_example + 10 mixed chunks >= 50%
        c_examples = [c for c in code_chunks if c.get("metadata", {}).get("content_type") == "code_example"]
        c_mixed = [c for c in code_chunks if c.get("metadata", {}).get("content_type") != "code_example"]
        rng.shuffle(c_examples)
        rng.shuffle(c_mixed)
        sampled_code = c_examples[:30] + c_mixed[:10]

        return {"text": sampled_text, "code": sampled_code}

class CheckpointManager:
    """Quản lý checkpoint an toàn, chống ghi đè và hỗ trợ khôi phục cross-session."""
    def __init__(self, checkpoint_dir: Path, input_checkpoint_dirs: Optional[List[Path]] = None, dataset_fingerprint: Optional[str] = None):
        self.checkpoint_dir = Path(checkpoint_dir)
        self.checkpoint_dir.mkdir(parents=True, exist_ok=True)
        self.input_dirs = [Path(p) for p in (input_checkpoint_dirs or []) if Path(p).exists()]
        self.dataset_fingerprint = dataset_fingerprint

    def load_completed_ids(self, model_name: str, expected_dimension: Optional[int] = None) -> Set[str]:
        completed = set()
        dirs = [self.checkpoint_dir / model_name]
        for inp in self.input_dirs:
            cand = inp / model_name if (inp / model_name).exists() else inp
            if cand.is_dir():
                dirs.append(cand)

        for d in dirs:
            if not d.exists():
                continue
            for f in d.glob("batch_*.jsonl"):
                try:
                    with open(f, "r", encoding="utf-8") as jf:
                        for line in jf:
                            if line.strip():
                                row = json.loads(line)
                                if expected_dimension and row.get("embedding_dimension") != expected_dimension:
                                    raise ValueError(f"Dimension mismatch in {f}")
                                cid = row.get("chunk_id")
                                if cid:
                                    completed.add(cid)
                except Exception as e:
                    print(f"⚠️ Cảnh báo đọc file {f}: {e}")
        return completed

    def save_batch(self, model_name: str, batch_idx: int, records: List[dict], dimension: int, model_id: str):
        mdir = self.checkpoint_dir / model_name
        mdir.mkdir(parents=True, exist_ok=True)
        # Tìm file index không trùng lặp
        target_idx = batch_idx
        fpath = mdir / f"batch_{target_idx:05d}.jsonl"
        while fpath.exists():
            target_idx += 1
            fpath = mdir / f"batch_{target_idx:05d}.jsonl"
        tmp_path = mdir / f"batch_{target_idx:05d}.jsonl.tmp"
        with open(tmp_path, "w", encoding="utf-8") as out:
            for r in records:
                out.write(json.dumps(r, ensure_ascii=False) + "\n")
        tmp_path.replace(fpath)
        # Cập nhật manifest
        self._update_manifest(mdir, model_name, target_idx, len(records), fpath.name, dimension, model_id)

    def _update_manifest(self, mdir: Path, model_name: str, batch_idx: int, count: int, fname: str, dimension: int, model_id: str):
        mfile = mdir / "checkpoint_manifest.json"
        mdata = {
            "model_name": model_name,
            "model_id": model_id,
            "dimension": dimension,
            "dataset_fingerprint": self.dataset_fingerprint,
            "last_updated": datetime.datetime.now(datetime.timezone.utc).isoformat(),
            "batches": []
        }
        if mfile.exists():
            try:
                with open(mfile, "r", encoding="utf-8") as mf:
                    mdata = json.load(mf)
            except Exception:
                pass
        mdata["batches"].append({"batch_idx": batch_idx, "file": fname, "records": count})
        tmp_m = mdir / "checkpoint_manifest.json.tmp"
        with open(tmp_m, "w", encoding="utf-8") as out:
            json.dump(mdata, out, ensure_ascii=False, indent=2)
        tmp_m.replace(mfile)

    def consolidate(self, model_name: str, output_file: Path) -> int:
        output_file = Path(output_file)
        output_file.parent.mkdir(parents=True, exist_ok=True)
        dirs = [self.checkpoint_dir / model_name]
        for inp in self.input_dirs:
            cand = inp / model_name if (inp / model_name).exists() else inp
            if cand.is_dir():
                dirs.append(cand)
        seen = set()
        total = 0
        with open(output_file, "w", encoding="utf-8") as out:
            for d in dirs:
                if not d.exists():
                    continue
                for bf in sorted(list(d.glob("batch_*.jsonl"))):
                    with open(bf, "r", encoding="utf-8") as inf:
                        for line in inf:
                            if line.strip():
                                r = json.loads(line)
                                cid = r.get("chunk_id")
                                if cid and cid not in seen:
                                    seen.add(cid)
                                    out.write(line)
                                    total += 1
        return total

class VectorValidator:
    """Kiểm tra QA chất lượng vector và schema metadata."""
    REQUIRED_FIELDS = [
        "source_file", "section_title", "section_path", "section_depth",
        "content_type", "language", "token_count", "has_code", "has_table", "has_equation"
    ]
    def __init__(self, expected_dim: int, l2_tolerance: float = 1e-3):
        self.expected_dim = expected_dim
        self.l2_tolerance = l2_tolerance

    def validate(self, record: dict) -> Tuple[bool, List[str]]:
        errors = []
        vec = record.get("vector", [])
        if not vec or len(vec) != self.expected_dim:
            errors.append(f"Dimension mismatch: expected {self.expected_dim}, got {len(vec)}")
        if any(math.isnan(v) or math.isinf(v) for v in vec):
            errors.append("Vector contains NaN/Inf")
        norm = math.sqrt(sum(v*v for v in vec))
        if abs(norm - 1.0) > self.l2_tolerance:
            errors.append(f"L2 norm deviation: {norm:.6f} != 1.0")
        meta = record.get("metadata", {})
        for rf in self.REQUIRED_FIELDS:
            if rf not in meta:
                errors.append(f"Missing metadata field: {rf}")
        return len(errors) == 0, errors

print("✅ Framework components đã được định nghĩa thành công!")


In [ ]:
# =============================================================================
# 5. NẠP DỮ LIỆU & LẤY MẪU PILOT ĐẠI DIỆN (160 CHUNKS)
# =============================================================================
with open(INPUT_CHUNK_FILE, "r", encoding="utf-8") as f:
    loaded_data = json.load(f)

# Nếu input là file pilot 160 chunks có sẵn
if len(loaded_data) == 160:
    print("📂 Phát hiện file input đã là tập Pilot 160 chunks chuẩn!")
    router = ModalityRouter(code_ratio_threshold=0.50)
    routed = router.route(loaded_data)
    pilot_text = routed[Modality.TEXT]
    pilot_code = routed[Modality.CODE]
else:
    # Nếu input là all_chunks.json (5,230 chunks)
    print(f"📊 Nạp dataset tổng: {len(loaded_data):,} chunks. Tiến hành định tuyến Modality...")
    router = ModalityRouter(code_ratio_threshold=0.50)
    partitions = router.route(loaded_data)
    print(f"  • Toàn bộ TEXT: {len(partitions[Modality.TEXT]):,} chunks (Kỳ vọng: 4.293)")
    print(f"  • Toàn bộ CODE: {len(partitions[Modality.CODE]):,} chunks (Kỳ vọng: 937)")
    assert len(partitions[Modality.TEXT]) == 4293, "Lỗi đếm TEXT!"
    assert len(partitions[Modality.CODE]) == 937, "Lỗi đếm CODE!"
    print("🎉 Đối soát Modality toàn bộ dataset 100% chính xác!")

    sampler = PilotSampler(seed=42, router=router)
    pilot_dict = sampler.sample(loaded_data, text_limit=120, code_limit=40)
    pilot_text = pilot_dict["text"]
    pilot_code = pilot_dict["code"]

print(f"\n🎯 TẬP MẪU PILOT GIAI ĐOẠN 3:")
print(f"  • Pilot A (TEXT): {len(pilot_text)} chunks (đại diện đa dạng các nguồn tài liệu)")
print(f"  • Pilot B (CODE): {len(pilot_code)} chunks (gồm cả code_example và mixed chunks >= 50%)")
print(f"  • Tổng số Pilot: {len(pilot_text) + len(pilot_code)} chunks")

assert len(pilot_text) == 120, f"Pilot A phải có đúng 120 chunks, got {len(pilot_text)}"
assert len(pilot_code) == 40, f"Pilot B phải có đúng 40 chunks, got {len(pilot_code)}"
print("✅ Bộ mẫu Pilot 160 chunks đã sẵn sàng!")


In [ ]:
# =============================================================================
# 6. PILOT A: QWEN3-EMBEDDING-0.6B (120 TEXT CHUNKS)
# =============================================================================
from transformers import AutoModel, AutoTokenizer
from tqdm import tqdm

MODEL_ID_QWEN = "Qwen/Qwen3-Embedding-0.6B"
DIM_QWEN = 1024
BATCH_SIZE_QWEN = 32  # Có thể điều chỉnh: 16 hoặc 32

print(f"=== BẮT ĐẦU NẠP MODEL PILOT A: {MODEL_ID_QWEN} ===")
start_load_qwen = time.time()
tokenizer_qwen = AutoTokenizer.from_pretrained(MODEL_ID_QWEN, padding_side="left")
model_qwen = AutoModel.from_pretrained(
    MODEL_ID_QWEN,
    torch_dtype=torch.float16,
    device_map="auto"
).eval()
load_time_qwen = time.time() - start_load_qwen
print(f"✅ Nạp Qwen3 hoàn tất trong {load_time_qwen:.2f}s")

def qwen3_last_token_pool(last_hidden_states, attention_mask):
    left_padding = (attention_mask[:, -1].sum() == attention_mask.shape[0])
    if left_padding:
        return last_hidden_states[:, -1]
    else:
        sequence_lengths = attention_mask.sum(dim=1) - 1
        batch_size = last_hidden_states.shape[0]
        return last_hidden_states[torch.arange(batch_size, device=last_hidden_states.device), sequence_lengths]

ckpt_mgr = CheckpointManager(CHECKPOINT_DIR, dataset_fingerprint="pilot_v3")
completed_qwen = ckpt_mgr.load_completed_ids("qwen3_text", expected_dimension=DIM_QWEN)
pending_qwen = [c for c in pilot_text if c["id"] not in completed_qwen]
print(f"Pilot A: {len(pilot_text)} total -> {len(completed_qwen)} đã xong -> {len(pending_qwen)} cần chạy.")

validator_qwen = VectorValidator(expected_dim=DIM_QWEN)
start_infer_qwen = time.time()
torch.cuda.reset_peak_memory_stats()

num_batches_qwen = (len(pending_qwen) + BATCH_SIZE_QWEN - 1) // BATCH_SIZE_QWEN
qwen_valid_count = 0
qwen_failures = []

for b_idx in tqdm(range(num_batches_qwen), desc="Pilot A - Qwen3"): 
    batch_chunks = pending_qwen[b_idx * BATCH_SIZE_QWEN : (b_idx + 1) * BATCH_SIZE_QWEN]
    # Document embedding: Không thêm instruction prefix
    texts = [c.get("text", "") for c in batch_chunks]
    
    try:
        with torch.no_grad():
            inputs = tokenizer_qwen(texts, max_length=8192, padding=True, truncation=True, return_tensors="pt").to(model_qwen.device)
            outputs = model_qwen(**inputs)
            embeddings = qwen3_last_token_pool(outputs.last_hidden_state, inputs["attention_mask"])
            embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)
            vectors = embeddings.cpu().to(torch.float32).tolist()
    except torch.cuda.OutOfMemoryError:
        print("⚠️ CUDA OOM! Dọn cache và thử xử lý từng item nhỏ...")
        torch.cuda.empty_cache()
        vectors = []
        with torch.no_grad():
            for single_text in texts:
                s_inp = tokenizer_qwen([single_text], max_length=8192, padding=True, truncation=True, return_tensors="pt").to(model_qwen.device)
                s_out = model_qwen(**s_inp)
                s_emb = torch.nn.functional.normalize(qwen3_last_token_pool(s_out.last_hidden_state, s_inp["attention_mask"]), p=2, dim=1)
                vectors.append(s_emb.cpu().to(torch.float32).tolist()[0])

    batch_records = []
    for chunk, vec in zip(batch_chunks, vectors):
        rec = {
            "chunk_id": chunk["id"],
            "source_file": chunk.get("metadata", {}).get("source_file", ""),
            "section_path": chunk.get("metadata", {}).get("section_path", ""),
            "modality": "text",
            "model_name": MODEL_ID_QWEN,
            "model_version": "1.0",
            "embedding_dimension": len(vec),
            "vector": vec,
            "token_count": chunk.get("metadata", {}).get("token_count", 0),
            "content_preview": chunk.get("text", "")[:120].replace("\n", " "),
            "metadata": chunk.get("metadata", {})
        }
        is_val, errs = validator_qwen.validate(rec)
        if is_val:
            batch_records.append(rec)
        else:
            qwen_failures.append({"id": chunk["id"], "errors": errs})

    if batch_records:
        ckpt_mgr.save_batch("qwen3_text", b_idx, batch_records, DIM_QWEN, MODEL_ID_QWEN)
        qwen_valid_count += len(batch_records)

infer_time_qwen = time.time() - start_infer_qwen
peak_vram_qwen = torch.cuda.max_memory_allocated() / (1024 ** 3)

# Hợp nhất vector
qwen_output_file = EMBEDDINGS_DIR / "qwen3_text" / "pilot_qwen3_text_vectors.jsonl"
total_qwen = ckpt_mgr.consolidate("qwen3_text", qwen_output_file)

print(f"\n📊 KẾT QUẢ PILOT A (Qwen3 Text):")
print(f"  • Vector hợp lệ : {total_qwen}/120")
print(f"  • Lỗi QA        : {len(qwen_failures)}")
print(f"  • Thời gian chạy: {infer_time_qwen:.2f}s ({len(pending_qwen)/max(infer_time_qwen, 0.001):.1f} chunks/s)")
print(f"  • Peak VRAM thực: {peak_vram_qwen:.2f} GB")

# GIẢI PHÓNG TRIỆT ĐỂ VRAM TRƯỚC KHI NẠP MODEL TIẾP THEO
print("🧹 Đang giải phóng Qwen3 khỏi GPU VRAM...")
del model_qwen, tokenizer_qwen
torch.cuda.empty_cache()
gc.collect()
print(f"✅ Đã giải phóng hoàn toàn VRAM (Hiện tại: {torch.cuda.memory_allocated() / (1024**3):.2f} GB)")


In [ ]:
# =============================================================================
# 7. PILOT B: JINA-EMBEDDINGS-V2-BASE-CODE (40 CODE CHUNKS)
# =============================================================================
from sentence_transformers import SentenceTransformer

MODEL_ID_JINA = "jinaai/jina-embeddings-v2-base-code"
DIM_JINA = 768
BATCH_SIZE_JINA = 16  # Có thể điều chỉnh: 16 hoặc 32

print(f"=== BẮT ĐẦU NẠP MODEL PILOT B: {MODEL_ID_JINA} ===")
start_load_jina = time.time()
model_jina = SentenceTransformer(MODEL_ID_JINA, trust_remote_code=True, device="cuda")
load_time_jina = time.time() - start_load_jina
print(f"✅ Nạp Jina Code hoàn tất trong {load_time_jina:.2f}s")

completed_jina = ckpt_mgr.load_completed_ids("jina_code", expected_dimension=DIM_JINA)
pending_jina = [c for c in pilot_code if c["id"] not in completed_jina]
print(f"Pilot B: {len(pilot_code)} total -> {len(completed_jina)} đã xong -> {len(pending_jina)} cần chạy.")

validator_jina = VectorValidator(expected_dim=DIM_JINA)
start_infer_jina = time.time()
torch.cuda.reset_peak_memory_stats()

num_batches_jina = (len(pending_jina) + BATCH_SIZE_JINA - 1) // BATCH_SIZE_JINA
jina_valid_count = 0
jina_failures = []

for b_idx in tqdm(range(num_batches_jina), desc="Pilot B - Jina Code"):
    batch_chunks = pending_jina[b_idx * BATCH_SIZE_JINA : (b_idx + 1) * BATCH_SIZE_JINA]
    texts = [c.get("text", "") for c in batch_chunks]
    
    try:
        embeddings = model_jina.encode(
            texts,
            batch_size=BATCH_SIZE_JINA,
            normalize_embeddings=True,
            show_progress_bar=False
        ).tolist()
    except Exception as e:
        print(f"⚠️ Lỗi batch Jina ({e}), fallback sang batch size nhỏ...")
        torch.cuda.empty_cache()
        embeddings = model_jina.encode(
            texts,
            batch_size=4,
            normalize_embeddings=True,
            show_progress_bar=False
        ).tolist()

    batch_records = []
    for chunk, vec in zip(batch_chunks, embeddings):
        # Kiểm tra tính toàn vẹn ký tự thụt lề và cú pháp markdown
        raw_text = chunk.get("text", "")
        rec = {
            "chunk_id": chunk["id"],
            "source_file": chunk.get("metadata", {}).get("source_file", ""),
            "section_path": chunk.get("metadata", {}).get("section_path", ""),
            "modality": "code",
            "model_name": MODEL_ID_JINA,
            "model_version": "2.0",
            "embedding_dimension": len(vec),
            "vector": vec,
            "token_count": chunk.get("metadata", {}).get("token_count", 0),
            "content_preview": raw_text[:120].replace("\n", " "),
            "metadata": chunk.get("metadata", {})
        }
        is_val, errs = validator_jina.validate(rec)
        if is_val:
            batch_records.append(rec)
        else:
            jina_failures.append({"id": chunk["id"], "errors": errs})

    if batch_records:
        ckpt_mgr.save_batch("jina_code", b_idx, batch_records, DIM_JINA, MODEL_ID_JINA)
        jina_valid_count += len(batch_records)

infer_time_jina = time.time() - start_infer_jina
peak_vram_jina = torch.cuda.max_memory_allocated() / (1024 ** 3)

# Hợp nhất vector
jina_output_file = EMBEDDINGS_DIR / "jina_code" / "pilot_jina_code_vectors.jsonl"
total_jina = ckpt_mgr.consolidate("jina_code", jina_output_file)

print(f"\n📊 KẾT QUẢ PILOT B (Jina Code):")
print(f"  • Vector hợp lệ : {total_jina}/40")
print(f"  • Lỗi QA        : {len(jina_failures)}")
print(f"  • Thời gian chạy: {infer_time_jina:.2f}s ({len(pending_jina)/max(infer_time_jina, 0.001):.1f} chunks/s)")
print(f"  • Peak VRAM thực: {peak_vram_jina:.2f} GB")

# GIẢI PHÓNG VRAM
del model_jina
torch.cuda.empty_cache()
gc.collect()
print(f"✅ Đã giải phóng hoàn toàn VRAM (Hiện tại: {torch.cuda.memory_allocated() / (1024**3):.2f} GB)")


In [ ]:
# =============================================================================
# 8. TỔNG KẾT PILOT, XUẤT BÁO CÁO JSON & ĐÓNG GÓI BẢN PHÁT HÀNH
# =============================================================================
import tarfile

# Kiểm tra trùng lặp hoặc thiếu chunk ID
qwen_ids = set()
if qwen_output_file.exists():
    with open(qwen_output_file, "r", encoding="utf-8") as f:
        for l in f:
            if l.strip():
                qwen_ids.add(json.loads(l)["chunk_id"])

jina_ids = set()
if jina_output_file.exists():
    with open(jina_output_file, "r", encoding="utf-8") as f:
        for l in f:
            if l.strip():
                jina_ids.add(json.loads(l)["chunk_id"])

overlap_ids = qwen_ids.intersection(jina_ids)
assert len(overlap_ids) == 0, f"Phát hiện trùng lặp ID giữa hai model: {overlap_ids}"

# Đề xuất batch size cho lần chạy toàn diện tiếp theo
rec_bs_qwen = 32 if peak_vram_qwen < 8.0 else 16
rec_bs_jina = 32 if peak_vram_jina < 8.0 else 16

report_data = {
    "timestamp": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "pilot_target": 160,
    "pilot_a_qwen3": {
        "model_id": MODEL_ID_QWEN,
        "expected_chunks": 120,
        "completed_vectors": total_qwen,
        "dimension": DIM_QWEN,
        "load_time_sec": round(load_time_qwen, 2),
        "inference_time_sec": round(infer_time_qwen, 2),
        "peak_vram_gb": round(peak_vram_qwen, 2),
        "failures_count": len(qwen_failures),
        "recommended_batch_size": rec_bs_qwen
    },
    "pilot_b_jina": {
        "model_id": MODEL_ID_JINA,
        "expected_chunks": 40,
        "completed_vectors": total_jina,
        "dimension": DIM_JINA,
        "load_time_sec": round(load_time_jina, 2),
        "inference_time_sec": round(infer_time_jina, 2),
        "peak_vram_gb": round(peak_vram_jina, 2),
        "failures_count": len(jina_failures),
        "recommended_batch_size": rec_bs_jina
    },
    "overall": {
        "total_pilot_embedded": total_qwen + total_jina,
        "pilot_success_rate": round((total_qwen + total_jina) / 160 * 100, 2),
        "id_collision_count": len(overlap_ids)
    }
}

report_path = REPORTS_DIR / "kaggle_pilot_benchmark_report.json"
with open(report_path, "w", encoding="utf-8") as rf:
    json.dump(report_data, rf, ensure_ascii=False, indent=2)

print("📊 BÁO CÁO NGHIỆM THU PILOT GIAI ĐOẠN 3:")
print(json.dumps(report_data, indent=2))

# Đóng gói nén tar.gz tại /kaggle/working/
archive_path = WORKING_DIR / "ai_coding_tutor_pilot_bundle.tar.gz"
with tarfile.open(archive_path, "w:gz") as tar:
    tar.add(EMBEDDINGS_DIR, arcname="embeddings")
    tar.add(REPORTS_DIR, arcname="reports")
    tar.add(CHECKPOINT_DIR, arcname="checkpoints")

print(f"\n📦 ĐÃ ĐÓNG GÓI THÀNH CÔNG: {archive_path} ({archive_path.stat().st_size:,} bytes)")
print("👉 Bạn có thể tải file này trực tiếp từ tab 'Output' của Kaggle Notebook để xem xét!")
